In [ ]:
# Import standard libraries
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from tqdm import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Data and Vocabularies

In [ ]:
# Load data and utilities
from src.data_loader import load_data, create_dataloaders, create_bert_dataloaders
from src.utils import build_vocab, load_slot_vocab, load_intent_vocab, compute_slot_f1

# Load datasets
train_data = load_data('dataset/train')
val_data = load_data('dataset/valid')
test_data = load_data('dataset/test')

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")
print(f"\nSample data: {train_data[0]}")

In [ ]:
# Load vocabularies
intent_vocab = load_intent_vocab('dataset/vocab.intent')
slot_vocab = load_slot_vocab('dataset/vocab.slot')

# Build word vocabulary from training data only
train_sentences = [' '.join(item['words']) for item in train_data]
word_vocab = build_vocab(train_sentences, min_freq=1)

print(f"Intent classes: {len(intent_vocab)}")
print(f"Slot labels: {len(slot_vocab)}")
print(f"Vocabulary size: {len(word_vocab)}")
print(f"\nIntent labels: {list(intent_vocab.keys())}")

## 2. Define Training and Evaluation Functions

In [ ]:
def train_joint_model(
    model, 
    train_loader, 
    val_loader, 
    epochs=5, 
    lr=1e-3,
    slot_pad_idx=0,
    alpha=1.0,  # weight for intent loss
    beta=1.0,   # weight for slot loss
    clip_grad=5.0
):
    """
    Train a joint intent classification and slot filling model.
    
    Args:
        model: The joint NLU model
        train_loader: Training DataLoader
        val_loader: Validation DataLoader
        epochs: Number of training epochs
        lr: Learning rate
        slot_pad_idx: Padding index for slot labels (ignored in loss)
        alpha: Weight for intent classification loss
        beta: Weight for slot filling loss
        clip_grad: Maximum gradient norm for clipping
    """
    model = model.to(device)
    
    # Loss functions
    intent_criterion = nn.CrossEntropyLoss()
    slot_criterion = nn.CrossEntropyLoss(ignore_index=slot_pad_idx)
    
    optimizer = Adam(model.parameters(), lr=lr)
    
    history = {'train_loss': [], 'val_loss': [], 'intent_acc': [], 'slot_f1': []}
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        total_train_loss = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            intent_labels = batch['intent_label'].to(device)
            slot_labels = batch['slot_labels'].to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            intent_logits, slot_logits = model(input_ids, attention_mask)
            
            # Compute losses
            intent_loss = intent_criterion(intent_logits, intent_labels)
            # Reshape slot logits for cross entropy: (batch * seq_len, num_slots)
            slot_logits_flat = slot_logits.view(-1, slot_logits.size(-1))
            slot_labels_flat = slot_labels.view(-1)
            slot_loss = slot_criterion(slot_logits_flat, slot_labels_flat)
            
            # Combined loss
            total_loss = alpha * intent_loss + beta * slot_loss
            
            # Backward pass
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            
            total_train_loss += total_loss.item()
        
        avg_train_loss = total_train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)
        
        # Validation phase
        val_metrics = evaluate_joint_model(model, val_loader, slot_pad_idx, slot_vocab)
        history['val_loss'].append(val_metrics['total_loss'])
        history['intent_acc'].append(val_metrics['intent_accuracy'])
        history['slot_f1'].append(val_metrics['slot_f1'])
        
        print(f"Epoch {epoch+1}/{epochs}:")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {val_metrics['total_loss']:.4f}")
        print(f"  Intent Accuracy: {val_metrics['intent_accuracy']:.4f}")
        print(f"  Slot F1: {val_metrics['slot_f1']:.4f}")
        print(f"  Joint Accuracy: {val_metrics['joint_accuracy']:.4f}")
        print()
    
    return history

In [ ]:
def evaluate_joint_model(model, data_loader, slot_pad_idx=0, slot_vocab=None):
    """
    Evaluate a joint intent classification and slot filling model.
    
    Returns:
        Dictionary with metrics: total_loss, intent_accuracy, slot_f1, joint_accuracy
    """
    model = model.to(device)
    model.eval()
    
    intent_criterion = nn.CrossEntropyLoss()
    slot_criterion = nn.CrossEntropyLoss(ignore_index=slot_pad_idx)
    
    total_loss = 0
    intent_correct = 0
    intent_total = 0
    joint_correct = 0
    
    all_slot_preds = []
    all_slot_labels = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            intent_labels = batch['intent_label'].to(device)
            slot_labels = batch['slot_labels'].to(device)
            
            # Forward pass
            intent_logits, slot_logits = model(input_ids, attention_mask)
            
            # Compute losses
            intent_loss = intent_criterion(intent_logits, intent_labels)
            slot_logits_flat = slot_logits.view(-1, slot_logits.size(-1))
            slot_labels_flat = slot_labels.view(-1)
            slot_loss = slot_criterion(slot_logits_flat, slot_labels_flat)
            total_loss += (intent_loss + slot_loss).item()
            
            # Intent predictions
            intent_preds = torch.argmax(intent_logits, dim=1)
            intent_correct += (intent_preds == intent_labels).sum().item()
            intent_total += intent_labels.size(0)
            
            # Slot predictions
            slot_preds = torch.argmax(slot_logits, dim=2)  # (batch, seq_len)
            
            # Collect for F1 computation
            for i in range(slot_preds.size(0)):
                pred_seq = slot_preds[i].cpu().tolist()
                label_seq = slot_labels[i].cpu().tolist()
                all_slot_preds.append(pred_seq)
                all_slot_labels.append(label_seq)
                
                # Joint accuracy: both intent and all slots correct
                intent_match = (intent_preds[i] == intent_labels[i]).item()
                # Compare only non-padding slots
                slot_match = all(
                    p == l for p, l in zip(pred_seq, label_seq) 
                    if l != slot_pad_idx
                )
                if intent_match and slot_match:
                    joint_correct += 1
    
    # Compute metrics
    avg_loss = total_loss / len(data_loader)
    intent_accuracy = intent_correct / intent_total
    joint_accuracy = joint_correct / intent_total
    
    # Compute slot F1
    if slot_vocab is not None:
        _, _, slot_f1 = compute_slot_f1(
            all_slot_preds, 
            all_slot_labels, 
            slot_vocab, 
            ignore_index=slot_pad_idx
        )
    else:
        slot_f1 = 0.0
    
    return {
        'total_loss': avg_loss,
        'intent_accuracy': intent_accuracy,
        'slot_f1': slot_f1,
        'joint_accuracy': joint_accuracy
    }

## 3. Model 1: Encoder-Decoder with Learned Embeddings

In [ ]:
from src.models import EncoderDecoderNLU

# Create DataLoaders for Model 1 & 2
MAX_LEN = 50
BATCH_SIZE = 32

train_loader, val_loader, test_loader = create_dataloaders(
    train_data, val_data, test_data,
    word_vocab, slot_vocab, intent_vocab,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Initialize Model 1
model1 = EncoderDecoderNLU(
    vocab_size=len(word_vocab),
    embedding_dim=300,
    hidden_dim=128,
    num_intents=len(intent_vocab),
    num_slots=len(slot_vocab),
    n_layers=1,
    dropout=0.3,
    pad_idx=word_vocab['<PAD>']
)

print(f"Model 1 parameters: {sum(p.numel() for p in model1.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model1.parameters() if p.requires_grad):,}")

In [ ]:
# Train Model 1
history1 = train_joint_model(
    model1, 
    train_loader, 
    val_loader,
    epochs=5,
    lr=1e-3,
    slot_pad_idx=slot_vocab['<PAD>']
)

In [ ]:
# Evaluate Model 1 on test set
print("Model 1 - Test Set Evaluation:")
test_metrics1 = evaluate_joint_model(model1, test_loader, slot_vocab['<PAD>'], slot_vocab)
print(f"  Intent Accuracy: {test_metrics1['intent_accuracy']:.4f}")
print(f"  Slot F1: {test_metrics1['slot_f1']:.4f}")
print(f"  Joint Accuracy: {test_metrics1['joint_accuracy']:.4f}")

## 4. Model 2: Transformer with Multi-Head Attention

In [ ]:
from src.models import TransformerNLU

# Initialize Model 2
model2 = TransformerNLU(
    vocab_size=len(word_vocab),
    d_model=256,  # Smaller model for faster training
    nhead=8,
    num_encoder_layers=4,
    dim_feedforward=1024,
    num_intents=len(intent_vocab),
    num_slots=len(slot_vocab),
    max_len=MAX_LEN,
    dropout=0.1,
    pad_idx=word_vocab['<PAD>']
)

print(f"Model 2 parameters: {sum(p.numel() for p in model2.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model2.parameters() if p.requires_grad):,}")

In [ ]:
# Train Model 2
history2 = train_joint_model(
    model2, 
    train_loader, 
    val_loader,
    epochs=5,
    lr=5e-4,  # Lower LR for transformer
    slot_pad_idx=slot_vocab['<PAD>']
)

In [ ]:
# Evaluate Model 2 on test set
print("Model 2 - Test Set Evaluation:")
test_metrics2 = evaluate_joint_model(model2, test_loader, slot_vocab['<PAD>'], slot_vocab)
print(f"  Intent Accuracy: {test_metrics2['intent_accuracy']:.4f}")
print(f"  Slot F1: {test_metrics2['slot_f1']:.4f}")
print(f"  Joint Accuracy: {test_metrics2['joint_accuracy']:.4f}")

## 5. Model 3: Encoder-Decoder with Frozen BERT Embeddings

In [ ]:
from transformers import BertTokenizer
from src.models import BertEncoderDecoderNLU

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

# Create BERT DataLoaders with subword alignment
bert_train_loader, bert_val_loader, bert_test_loader = create_bert_dataloaders(
    train_data, val_data, test_data,
    tokenizer, slot_vocab, intent_vocab,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN
)

print(f"BERT Training batches: {len(bert_train_loader)}")

In [ ]:
# Initialize Model 3
model3 = BertEncoderDecoderNLU(
    hidden_dim=128,
    num_intents=len(intent_vocab),
    num_slots=len(slot_vocab),
    n_layers=1,
    dropout=0.3,
    bert_model_name='bert-base-cased'
)

total_params = sum(p.numel() for p in model3.parameters())
trainable_params = sum(p.numel() for p in model3.parameters() if p.requires_grad)
print(f"Model 3 total parameters: {total_params:,}")
print(f"Model 3 trainable parameters: {trainable_params:,}")
print(f"BERT parameters (frozen): {total_params - trainable_params:,}")

In [ ]:
# For Model 3, we need to use -100 as ignore index (for subword alignment)
def train_bert_model(model, train_loader, val_loader, epochs=5, lr=1e-3):
    """Training function for BERT model with -100 ignore index."""
    model = model.to(device)
    
    intent_criterion = nn.CrossEntropyLoss()
    slot_criterion = nn.CrossEntropyLoss(ignore_index=-100)  # Ignore special tokens
    
    optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            intent_labels = batch['intent_label'].to(device)
            slot_labels = batch['slot_labels'].to(device)
            
            optimizer.zero_grad()
            
            intent_logits, slot_logits = model(input_ids, attention_mask)
            
            intent_loss = intent_criterion(intent_logits, intent_labels)
            slot_logits_flat = slot_logits.view(-1, slot_logits.size(-1))
            slot_labels_flat = slot_labels.view(-1)
            slot_loss = slot_criterion(slot_logits_flat, slot_labels_flat)
            
            total_loss = intent_loss + slot_loss
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            
            total_train_loss += total_loss.item()
        
        # Validation
        model.eval()
        val_intent_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                intent_labels = batch['intent_label'].to(device)
                
                intent_logits, _ = model(input_ids, attention_mask)
                intent_preds = torch.argmax(intent_logits, dim=1)
                val_intent_correct += (intent_preds == intent_labels).sum().item()
                val_total += intent_labels.size(0)
        
        print(f"Epoch {epoch+1}: Train Loss = {total_train_loss/len(train_loader):.4f}, "
              f"Val Intent Acc = {val_intent_correct/val_total:.4f}")

In [ ]:
# Train Model 3
train_bert_model(model3, bert_train_loader, bert_val_loader, epochs=5, lr=1e-3)

In [ ]:
# Evaluate Model 3 on test set
def evaluate_bert_model(model, data_loader):
    """Evaluate BERT model."""
    model = model.to(device)
    model.eval()
    
    intent_correct = 0
    slot_correct = 0
    slot_total = 0
    total = 0
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Testing"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            intent_labels = batch['intent_label'].to(device)
            slot_labels = batch['slot_labels'].to(device)
            
            intent_logits, slot_logits = model(input_ids, attention_mask)
            
            # Intent accuracy
            intent_preds = torch.argmax(intent_logits, dim=1)
            intent_correct += (intent_preds == intent_labels).sum().item()
            total += intent_labels.size(0)
            
            # Slot accuracy (only non-ignored positions)
            slot_preds = torch.argmax(slot_logits, dim=2)
            mask = slot_labels != -100
            slot_correct += ((slot_preds == slot_labels) & mask).sum().item()
            slot_total += mask.sum().item()
    
    print(f"Intent Accuracy: {intent_correct/total:.4f}")
    print(f"Slot Accuracy: {slot_correct/slot_total:.4f}")

print("Model 3 - Test Set Evaluation:")
evaluate_bert_model(model3, bert_test_loader)

## 6. Model Comparison Summary

In [ ]:
import pandas as pd

# Create comparison table
comparison = pd.DataFrame({
    'Model': ['Encoder-Decoder (Learned)', 'Transformer', 'BERT + Encoder-Decoder'],
    'Parameters': [
        f"{sum(p.numel() for p in model1.parameters()):,}",
        f"{sum(p.numel() for p in model2.parameters()):,}",
        f"{sum(p.numel() for p in model3.parameters() if p.requires_grad):,} (trainable)"
    ],
    'Intent Acc': [
        f"{test_metrics1['intent_accuracy']:.4f}" if 'test_metrics1' in dir() else 'N/A',
        f"{test_metrics2['intent_accuracy']:.4f}" if 'test_metrics2' in dir() else 'N/A',
        'See above'
    ],
    'Slot F1': [
        f"{test_metrics1['slot_f1']:.4f}" if 'test_metrics1' in dir() else 'N/A',
        f"{test_metrics2['slot_f1']:.4f}" if 'test_metrics2' in dir() else 'N/A',
        'See above'
    ]
})

print("Model Comparison:")
print(comparison.to_string(index=False))

## 7. Save Models

In [ ]:
# Save model weights
torch.save(model1.state_dict(), 'encoder_decoder_nlu.pth')
torch.save(model2.state_dict(), 'transformer_nlu.pth')

# For Model 3, only save GRU weights (BERT is frozen)
model3_state = {
    'encoder': model3.encoder.state_dict(),
    'decoder': model3.decoder.state_dict(),
    'encoder_to_decoder': model3.encoder_to_decoder.state_dict(),
    'intent_classifier': model3.intent_classifier.state_dict(),
    'slot_classifier': model3.slot_classifier.state_dict(),
}
torch.save(model3_state, 'bert_encoder_decoder_nlu.pth')

print("Models saved successfully!")